# 01 — Generate Synthetic Casino Event Data

Generates **synthetic** slot machine / gaming floor event data and writes it as
JSON files to a Unity Catalog Volume. This simulates data landing in cloud
storage — the same pattern you'd see with files arriving from Kafka Connect,
an ETL tool, or any file-based pipeline.

All data in this demo is **fake / randomly generated** and does not represent
any real casino, player, or organization.

In [1]:
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv
from faker import Faker
import os
import json
import random
import uuid
from datetime import datetime, timedelta

load_dotenv(dotenv_path="../.env")

spark = DatabricksSession.builder.serverless().getOrCreate()
w = WorkspaceClient()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "streaming_demo")

fake = Faker()
Faker.seed(42)
random.seed(42)

## Create a UC Volume for raw JSON files

In [2]:
VOLUME_NAME = "casino_raw_events"
VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{VOLUME_NAME}"

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}.{VOLUME_NAME}
""")

print(f"Volume ready: {VOLUME_PATH}")

Volume ready: /Volumes/alexander_booth/streaming_demo/casino_raw_events


## Define the synthetic event generator

Each event represents a single play on a casino gaming machine.

In [3]:
CASINO_FLOORS = ["Main Floor", "High Roller", "VIP Lounge", "Penny Lane", "Sports Zone"]
GAME_TYPES = ["Slots", "Video Poker", "Electronic Roulette", "Electronic Blackjack", "Keno"]
MACHINE_IDS = [f"MCH-{i:04d}" for i in range(1, 51)]  # 50 machines
PLAYER_IDS = [f"PLR-{fake.unique.random_int(min=10000, max=99999)}" for _ in range(100)]  # 100 players


def generate_event(base_time):
    """Generate a single synthetic casino gaming event."""
    game_type = random.choice(GAME_TYPES)

    # Bet amounts vary by game type
    bet_ranges = {
        "Slots": (0.25, 50.00),
        "Video Poker": (1.00, 100.00),
        "Electronic Roulette": (5.00, 500.00),
        "Electronic Blackjack": (10.00, 1000.00),
        "Keno": (1.00, 20.00),
    }
    min_bet, max_bet = bet_ranges[game_type]
    bet_amount = round(random.uniform(min_bet, max_bet), 2)

    # House edge: most plays lose, some win small, rare big wins
    outcome_roll = random.random()
    if outcome_roll < 0.55:       # 55% lose everything
        win_amount = 0.0
    elif outcome_roll < 0.80:     # 25% win small (0.5x - 2x bet)
        win_amount = round(bet_amount * random.uniform(0.5, 2.0), 2)
    elif outcome_roll < 0.95:     # 15% win medium (2x - 5x bet)
        win_amount = round(bet_amount * random.uniform(2.0, 5.0), 2)
    else:                         # 5% jackpot (5x - 20x bet)
        win_amount = round(bet_amount * random.uniform(5.0, 20.0), 2)

    event_time = base_time + timedelta(seconds=random.randint(0, 59))

    return {
        "event_id": str(uuid.uuid4()),
        "machine_id": random.choice(MACHINE_IDS),
        "casino_floor": random.choice(CASINO_FLOORS),
        "game_type": game_type,
        "player_id": random.choice(PLAYER_IDS),
        "bet_amount": bet_amount,
        "win_amount": win_amount,
        "event_timestamp": event_time.isoformat(),
    }


# Preview a sample event
sample = generate_event(datetime.now())
print(json.dumps(sample, indent=2))

{
  "event_id": "e070d00a-53da-40f6-9a15-5066510e2f40",
  "machine_id": "MCH-0009",
  "casino_floor": "Main Floor",
  "game_type": "Slots",
  "player_id": "PLR-31417",
  "bet_amount": 1.49,
  "win_amount": 0.0,
  "event_timestamp": "2026-03-27T11:38:41.586189"
}


## Write events as JSON files to the Volume

We write 10 batches of 50 events each (500 total), simulating data arriving
over time. Each batch is a separate JSON file — exactly what Auto Loader
is designed to pick up incrementally.

In [4]:
NUM_BATCHES = 10
EVENTS_PER_BATCH = 50
base_time = datetime(2026, 3, 25, 10, 0, 0)  # start time for events

total_events = 0

for batch_num in range(NUM_BATCHES):
    batch_time = base_time + timedelta(minutes=batch_num)
    events = [generate_event(batch_time) for _ in range(EVENTS_PER_BATCH)]

    # Write as newline-delimited JSON (one JSON object per line)
    ndjson_content = "\n".join(json.dumps(e) for e in events)
    file_path = f"{VOLUME_PATH}/batch_{batch_num:03d}.json"

    w.files.upload(
        file_path=file_path,
        contents=ndjson_content.encode("utf-8"),
        overwrite=True,
    )
    total_events += len(events)
    print(f"  Wrote batch {batch_num:03d} → {len(events)} events")

print(f"\nTotal: {total_events} synthetic events across {NUM_BATCHES} files")
print(f"Volume path: {VOLUME_PATH}")

  Wrote batch 000 → 50 events
  Wrote batch 001 → 50 events
  Wrote batch 002 → 50 events
  Wrote batch 003 → 50 events
  Wrote batch 004 → 50 events
  Wrote batch 005 → 50 events
  Wrote batch 006 → 50 events
  Wrote batch 007 → 50 events
  Wrote batch 008 → 50 events
  Wrote batch 009 → 50 events

Total: 500 synthetic events across 10 files
Volume path: /Volumes/alexander_booth/streaming_demo/casino_raw_events


## Verify files landed

In [5]:
files = list(w.files.list_directory_contents(VOLUME_PATH))
print(f"{len(files)} files in {VOLUME_PATH}:")
for f in files[:5]:
    print(f"  {f.path}")
if len(files) > 5:
    print(f"  ... and {len(files) - 5} more")

10 files in /Volumes/alexander_booth/streaming_demo/casino_raw_events:
  /Volumes/alexander_booth/streaming_demo/casino_raw_events/batch_000.json
  /Volumes/alexander_booth/streaming_demo/casino_raw_events/batch_001.json
  /Volumes/alexander_booth/streaming_demo/casino_raw_events/batch_002.json
  /Volumes/alexander_booth/streaming_demo/casino_raw_events/batch_003.json
  /Volumes/alexander_booth/streaming_demo/casino_raw_events/batch_004.json
  ... and 5 more
